In [ ]:
import os
import seaborn as sns
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import contextily as ctx
import warnings
warnings.filterwarnings('ignore')
import sys
# sys.path.append("/path/to/sinkhole-risk-china/code")# project code path
sys.path.append("/path/to/sinkhole-risk-china/code")# project code path
from mgtwr.sel_bw import Sel_BW
from mgtwr.model import Model
import multiprocessing as mp
import psutil
from sklearn.metrics import r2_score
from shapely.geometry import Point
from pathlib import Path


In [ ]:
ssp = "hist"  # TODO: 'hist' / 'ssp1' / 'ssp2' / 'ssp3' / 'ssp4' / 'ssp5'
ssp_time = "2020"  # TODO: '2020' / '2040' / '2060' / '2080' / '2100'

In [ ]:
resolution = "10km"  # TODO: Manually change the resolution to '0.1km' / '1km' / '3km' / '5km' / '10km' / '20km' as needed
home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
data_base_path = os.path.join(base_path, "data")

path_name = f"Points_China_all_{resolution}"
print(path_name)
# ✅ Output directory: base_path/output/path_name
out_dir = os.path.join(base_path, "outputs", path_name, ssp, ssp_time)
print(f"out_dir:{out_dir}")
os.makedirs(out_dir, exist_ok=True)


Implementation note.

In [ ]:
# Data loading
data_path = os.path.join(data_base_path, "Extracted_HAVE_future", "Positive_Negative_balanced", "AllFeatures_Positive_Negative_balanced_25366_ssp1_cleaned.csv")
# data_path = os.path.join(base_path, "data", "ML", "all_set.csv")
df = pd.read_csv(data_path)

# Select the required characteristics
features = [
            'Distance_to_karst',
            'Depth_to_Bedrock',
            'Distance_to_Fault_m',
            'UrbanFrac_hist_2000_2010_2020',
            'PopTotal_hist_2000_2010_2020',
            'ImperviousIndex_hist_2000_2010_2020',
            'LAI_hist_2000_2010_2020',
            'Precip_hist_2000_2010_2020',
            'WTD_hist_2000_2010_2020',
            'HDS_hist_2000_2010_2020',

            'Tas_hist_2000_2010_2020',
            # 'Tasmax_hist_2000_2010_2020',
            # 'Tasmin_hist_2000_2010_2020',
            'Huss_hist_2000_2010_2020',
            ]

X = df[features].values
y = df['Disaster'].values.reshape(-1, 1)  # Reshape into a 2D column matrix
print("Shape of independent variable X:", X.shape)
print("Shape of dependent variable y:", y.shape)
# Create geographic coordinate system
geometry = [Point(lon, lat) for lon, lat in zip(df['Longitude'], df['Latitude'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

# Convert to projected coordinate system (using isometric projection)
gdf = gdf.to_crs("EPSG:3857")  # Web Mercator projection
coords = np.array([(pt.x, pt.y) for pt in gdf.geometry])
print("Shape of coordinates coords:", coords.shape)

print("\\n✅ features corresponding data preview of the first 5 lines:")
display(df[features].head())

Implementation note.

In [ ]:


# pre_data_path = os.path.join(data_base_path, "Extracted_HAVE_future", "Positive_Negative_balanced", "AllFeatures_Positive_Negative_balanced_25366_ssp5_cleaned.csv")
pre_data_path = os.path.join(data_base_path, "Extracted_HAVE_future", f"Points_China_all_{resolution}", f"AllFeatures_Points_China_all_{resolution}_{ssp}_cleaned.csv")
# pre_data_path = os.path.join(data_base_path, "Extracted_HAVE_future", "Positive_Negative_balanced", "AllFeatures_Positive_Negative_balanced_25366_ssp1_cleaned.csv")
pre_df = pd.read_csv(pre_data_path)
print("Columns (list):")
print(pre_df.columns.tolist())

###################################################################################
# ########################### Select the required characteristics##########################################
###################################################################################
# ---- Static features ----
static_features = [
    "Distance_to_karst",
    "Depth_to_Bedrock",
    "Distance_to_Fault_m",
]

# ---- Dynamic feature prefix ----
dynamic_prefixes = ["UrbanFrac", "PopTotal", "ImperviousIndex", "LAI", "Precip", "WTD", "HDS", "Tas", "Huss"]

def pick_dynamic_col(prefix: str, ssp: str, ssp_time: str) -> str:
    # 2020 or hist: use historical column
    if ssp == "hist" or ssp_time == "2020":
        return f"{prefix}_hist_2000_2010_2020"
    # 2040/2060/2080/2100: (t-20, t) 20
    t = int(ssp_time)
    return f"{prefix}_{t-20}_{t}"

pre_features = static_features + [pick_dynamic_col(p, ssp, ssp_time) for p in dynamic_prefixes]

# ---- Check for missing columns first to avoid direct KeyError ----
missing = [c for c in pre_features if c not in pre_df.columns]
if missing:
    print("❌ Missing columns:", missing)
    print("\n✅ Available columns (preview):", pre_df.columns.tolist()[:20], "...")
    raise KeyError("Some required features are not in the CSV columns.")

pre_X = pre_df[pre_features].values
print("✅ pre_features =", pre_features)
print("✅ pre_X shape =", pre_X.shape)
###################################################################################
###################################################################################
###################################################################################

pre_has_observed_labels = 'Disaster' in pre_df.columns
# If pre_df does not have a Disaster column, create it and fill it with 3
if not pre_has_observed_labels:
    pre_df['Disaster'] = 3

pre_y = pre_df['Disaster'].values.reshape(-1, 1)  # (n_samples, 1)

print("The shape of the argument pre_X:", pre_X.shape)
print("Shape of dependent variable pre_y:", pre_y.shape)
# Create geographic coordinate system
geometry = [Point(lon, lat) for lon, lat in zip(pre_df['Longitude'], pre_df['Latitude'])]
pre_gdf = gpd.GeoDataFrame(pre_df, geometry=geometry, crs="EPSG:4326")

# Convert to projected coordinate system (using isometric projection)
pre_gdf = pre_gdf.to_crs("EPSG:3857")  # Web Mercator projection
pre_coords1 = np.array([(pt.x, pt.y) for pt in pre_gdf.geometry])
pre_coords = np.hstack((pre_coords1, np.zeros((pre_coords1.shape[0], 1))))
print("Shape of coordinates pre_coords:", pre_coords.shape)

print("\\n✅ pre_features preview of the first 5 lines of corresponding data:")
display(pre_df[pre_features].head())

GWR

In [ ]:
# sel = Sel_BW(coords, y, X, mode='gwr', spherical=False, fixed=False) # spherical True
# print('searching...')
# bw, tau = sel.search(tau_min=1e-5, verbose=True, max_iter=100)
# print('bw:', bw, 'tau:', tau)

# import pickle

# # Save variables
# with open('variables_national_GWR.pkl', 'wb') as f:
#     pickle.dump({'bw': bw, 'tau': tau}, f)

In [ ]:
import pickle

# Read variables
with open('variables_national_GWR.pkl', 'rb') as f:
    loaded_vars = pickle.load(f)
    bw = loaded_vars['bw']
    tau = loaded_vars['tau']

In [ ]:
# gwr = Model(coords, y, X, bw=bw, mode='gwr', spherical=False, fixed=False) # spherical latitude and longitude coordinates are set to True
# gwr_results = gwr.fit()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_gwr_output_preview(values, title_prefix="GWR Raw Output", max_unique_bar=20, hist_bins=40):
    values = np.asarray(values, dtype=float).reshape(-1)
    valid_mask = np.isfinite(values)
    valid_values = values[valid_mask]
    if valid_values.size == 0:
        raise ValueError("There are no limited output values available for preview.")

    unique_values, counts = np.unique(valid_values, return_counts=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

    if unique_values.size <= max_unique_bar:
        x = np.arange(unique_values.size)
        axes[0].bar(x, counts, color="#8e7376", edgecolor="white")
        axes[0].set_xticks(x)
        axes[0].set_xticklabels([f"{v:.6g}" for v in unique_values], rotation=45, ha="right")
        axes[0].set_xlabel("Raw score")
        axes[0].set_ylabel("Count")
        axes[0].set_title(f"{title_prefix}: unique-value counts")
    else:
        axes[0].hist(valid_values, bins=hist_bins, color="#8e7376", edgecolor="white")
        axes[0].set_xlabel("Raw score")
        axes[0].set_ylabel("Count")
        axes[0].set_title(f"{title_prefix}: histogram")

    sorted_values = np.sort(valid_values)
    quantiles = np.linspace(0.0, 1.0, sorted_values.size)
    axes[1].plot(quantiles, sorted_values, color="#724e52", linewidth=1.5)
    axes[1].set_xlabel("Quantile")
    axes[1].set_ylabel("Raw score")
    axes[1].set_title(f"{title_prefix}: sorted curve")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"{title_prefix} finite count: {valid_values.size}")
    print(f"{title_prefix} raw-score range: [{valid_values.min():.6g}, {valid_values.max():.6g}]")
    print(f"{title_prefix} raw-score mean/std: {valid_values.mean():.6g} / {valid_values.std(ddof=0):.6g}")
    print(f"{title_prefix} unique value count: {unique_values.size}")
    if unique_values.size <= max_unique_bar:
        print("Unique raw scores:")
        for value, count in zip(unique_values, counts):
            print(f"  {value:.10g}: {count}")


plot_gwr_output_preview(gwr_results.predy, title_prefix="Training GWR raw outputs")


In [ ]:
# import pickle
# #####
# # Save to current folder
# with open("gwr_model_national_GWR.pkl", "wb") as f:
#     pickle.dump(
#         {"gwr": gwr, "gwr_results": gwr_results, "bw": bw},
#         f,
#         protocol=pickle.HIGHEST_PROTOCOL
#     )

In [ ]:
import pickle

with open("gwr_model_national_GWR.pkl", "rb") as f:
    saved_data = pickle.load(f)

gwr = saved_data["gwr"]
gwr_results = saved_data["gwr_results"]
bw = saved_data["bw"]
tau = saved_data.get("tau", 0.0)
# bw, tau = normalize_gwr_bw_tau(bw, tau)

print("Model reading successful!")
print(f"Bandwidth (bw):{bw}")
print(f"tau: {tau}")
print(f"GWR model object type:{type(gwr)}")
print(f"GWR :{type(gwr_results)}")


GWR——


In [ ]:
import pickle
from pathlib import Path
import mgtwr.gwr_sigmoid_utils as gwr_sigmoid_utils

LABEL_CUTOFF = 0.55
VALIDATION_SIZE = 0.2
RANDOM_STATE = 42
PROBABILITY_CLIP_Z = 6.0
CLASSIFICATION_ARTIFACTS_PATH = Path("gwr_classification_artifacts.pkl")

train_summary = gwr_sigmoid_utils.fit_gwr_probability_classifier(
    gwr_results.predy,
    y,
    label_cutoff=LABEL_CUTOFF,
    validation_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    clip_z=PROBABILITY_CLIP_Z,
)

train_raw_scores = train_summary["raw_scores"]
train_probabilities = train_summary["probabilities"]
Classification_results = train_summary["predicted_classes"]
transform_metadata = train_summary["transform_metadata"]
gwr_probability_threshold = train_summary["probability_threshold"]
gwr_classification_artifacts = train_summary["artifacts"]

with CLASSIFICATION_ARTIFACTS_PATH.open('wb') as f:
    pickle.dump(gwr_classification_artifacts, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\\n mgtwr.gwr_sigmoid_utils GWR 0-1")
print(f"Train the original output shape:{train_raw_scores.shape}")
print(f"The first few training raw outputs:{train_raw_scores[:10]}")
print(f"Conversion method:{transform_metadata['method']}")
print(f"center: {transform_metadata['center']:.6f}")
print(f"scale: {transform_metadata['scale']:.6f}")
print(f"clip_z: {transform_metadata['clip_z']:.2f}")
print(f":{float(train_probabilities.min()):.6f} ~ {float(train_probabilities.max()):.6f}")
print(f"Label to binary classification threshold:{LABEL_CUTOFF}")
print(f"Optimal probability threshold for validation set:{gwr_probability_threshold:.4f}")
print(f"Training set regression R2:{train_summary['train_regression_metrics']['r2']:.4f}")
print(f"Training set regression RMSE:{train_summary['train_regression_metrics']['rmse']:.4f}")
print(f"MAE:{train_summary['train_regression_metrics']['mae']:.4f}")
print(f"Accuracy:{train_summary['train_binary_metrics']['accuracy']:.4f}")
print(f"Training set ROC-AUC:{train_summary['train_binary_metrics']['roc_auc']:.4f}")
print(f"Validation set Accuracy:{train_summary['validation_binary_metrics']['accuracy']:.4f}")
print(f"Validation set ROC-AUC:{train_summary['validation_binary_metrics']['roc_auc']:.4f}")
print(f"Classified artifacts have been saved to:{CLASSIFICATION_ARTIFACTS_PATH}")


In [ ]:
# print(gwr_results.summary)

In [ ]:
# gwr_results.params

In [ ]:
# gwr_results.pvalues

GWR

In [ ]:
import pickle
# Processing step.
with open("gwr_model_national_GWR.pkl", "rb") as f:
    saved = pickle.load(f)

gwr = saved["gwr"]
gwr_results = saved.get("gwr_results", None)
bw = saved.get("bw", None)
######

In [ ]:
# import numpy as np
# from sklearn.metrics import r2_score

# y_pred_gwr, params = gwr.predict(pre_coords, pre_X)
# y_pred_gwr = np.asarray(y_pred_gwr, dtype=float).reshape(-1, 1)
# params = np.asarray(params, dtype=float)

In [ ]:
# ########################Truncate prediction set output (outlier handling) ####################################
# y_pred_gwr_unclipped = y_pred_gwr.copy()

# CLIP_MIN = -0.539341
# CLIP_MAX = 1.2834

# n_clipped_low = int(np.sum(y_pred_gwr_unclipped < CLIP_MIN))
# n_clipped_high = int(np.sum(y_pred_gwr_unclipped > CLIP_MAX))
# y_pred_gwr = np.clip(y_pred_gwr_unclipped, CLIP_MIN, CLIP_MAX)

# print(y_pred_gwr.shape)
# print(params.shape)
# print('Manual clipping range = ', CLIP_MIN, CLIP_MAX)
# print('Prediction raw-score range before clipping = ', float(np.nanmin(y_pred_gwr_unclipped)), float(np.nanmax(y_pred_gwr_unclipped)))
# print('Prediction raw-score range after clipping = ', float(np.nanmin(y_pred_gwr)), float(np.nanmax(y_pred_gwr)))
# print('Clipped below min count = ', n_clipped_low)
# print('Clipped above max count = ', n_clipped_high)

# if globals().get('pre_has_observed_labels', False):
#     print('R2 = ', float(r2_score(pre_y, y_pred_gwr)))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if 'plot_gwr_output_preview' not in globals():
    def plot_gwr_output_preview(values, title_prefix="GWR Raw Output", max_unique_bar=20, hist_bins=40, focus_pct=99.0):
        values = np.asarray(values, dtype=float).reshape(-1)
        valid_mask = np.isfinite(values)
        valid_values = values[valid_mask]
        if valid_values.size == 0:
            raise ValueError("There are no limited output values available for preview.")

        lower_pct = max(0.0, (100.0 - float(focus_pct)) / 2.0)
        upper_pct = min(100.0, 100.0 - lower_pct)
        score_min, score_max = np.nanpercentile(valid_values, [lower_pct, upper_pct])
        if (not np.isfinite(score_min)) or (not np.isfinite(score_max)):
            score_min = float(np.min(valid_values))
            score_max = float(np.max(valid_values))
        if score_min == score_max:
            pad = max(abs(float(score_min)) * 0.05, 1.0e-6)
            score_min -= pad
            score_max += pad

        unique_values, counts = np.unique(valid_values, return_counts=True)

        fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

        if unique_values.size <= max_unique_bar:
            x = np.arange(unique_values.size)
            axes[0].bar(x, counts, color="#8e7376", edgecolor="white")
            axes[0].set_xticks(x)
            axes[0].set_xticklabels([f"{v:.6g}" for v in unique_values], rotation=45, ha="right")
            axes[0].set_xlabel("Raw score")
            axes[0].set_ylabel("Count")
            axes[0].set_title(f"{title_prefix}: unique-value counts")
        else:
            display_values = valid_values[(valid_values >= score_min) & (valid_values <= score_max)]
            if display_values.size == 0:
                display_values = valid_values
            axes[0].hist(display_values, bins=np.linspace(score_min, score_max, hist_bins + 1), color="#8e7376", edgecolor="white")
            axes[0].set_xlim(score_min, score_max)
            axes[0].set_xlabel("Raw score")
            axes[0].set_ylabel("Count")
            axes[0].set_title(f"{title_prefix}: histogram (99% range)")

        sorted_values = np.sort(valid_values)
        quantiles = np.linspace(0.0, 1.0, sorted_values.size)
        axes[1].plot(quantiles, sorted_values, color="#724e52", linewidth=1.5)
        axes[1].set_xlabel("Quantile")
        axes[1].set_ylabel("Raw score")
        axes[1].set_title(f"{title_prefix}: sorted curve")
        axes[1].set_ylim(score_min, score_max)
        axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

        print(f"{title_prefix} finite count: {valid_values.size}")
        print(f"{title_prefix} raw-score range: [{valid_values.min():.6g}, {valid_values.max():.6g}]")
        print(f"{title_prefix} 99% display range: [{score_min:.6g}, {score_max:.6g}]")
        print(f"{title_prefix} raw-score mean/std: {valid_values.mean():.6g} / {valid_values.std(ddof=0):.6g}")
        print(f"{title_prefix} unique value count: {unique_values.size}")
        if unique_values.size <= max_unique_bar:
            print("Unique raw scores:")
            for value, count in zip(unique_values, counts):
                print(f"  {value:.10g}: {count}")

plot_gwr_output_preview(y_pred_gwr, title_prefix="Predicted GWR raw outputs")


/

In [ ]:
import pickle
from pathlib import Path
from sklearn.metrics import r2_score

# ()
pkl_path = Path(f"gwr_pre_{path_name}_{ssp}_{ssp_time}_results.pkl")
r2 = r2_score(pre_y, y_pred_gwr)

saved = {
    "y_pred_gwr": y_pred_gwr,
    "params": params,
    "r2": r2,}
with pkl_path.open("wb") as f:
    pickle.dump(saved, f, protocol=pickle.HIGHEST_PROTOCOL)


Read the current ssp and ssp_time data

In [ ]:
import pickle
from pathlib import Path
from sklearn.metrics import r2_score

# ()
pkl_path = Path(f"gwr_pre_{path_name}_{ssp}_{ssp_time}_results.pkl")

if pkl_path.exists():
    # Processing step.
    with pkl_path.open("rb") as f:
        saved = pickle.load(f)
    y_pred_gwr = saved["y_pred_gwr"]
    params = saved["params"]
    r2 = saved.get("r2", None)


hist 2020

In [ ]:
# pkl_path_hist_2020 = Path(f"gwr_pre_{path_name}_hist_2020_results.pkl")

# if pkl_path_hist_2020.exists():
# # Read temporary storage
#     with pkl_path_hist_2020.open("rb") as f:
#         saved_hist_2020 = pickle.load(f)
#     y_pred_gwr_hist_2020 = saved_hist_2020["y_pred_gwr"]
#     params_hist_2020 = saved_hist_2020["params"]
#     r2_hist_2020 = saved_hist_2020.get("r2", None)

GWR regression results are mapped to probability - prediction set


In [ ]:
import pickle
from pathlib import Path
import mgtwr.gwr_sigmoid_utils as gwr_sigmoid_utils

CLASSIFICATION_ARTIFACTS_PATH = Path("gwr_classification_artifacts.pkl")
if 'gwr_classification_artifacts' not in globals():
    with CLASSIFICATION_ARTIFACTS_PATH.open('rb') as f:
        gwr_classification_artifacts = pickle.load(f)

prediction_summary = gwr_sigmoid_utils.predict_gwr_probability_classifier(
    y_pred_gwr,
    gwr_classification_artifacts,
    y_true=pre_y if globals().get('pre_has_observed_labels', False) else None,
)

test_raw_scores = prediction_summary["raw_scores"]
test_probabilities = prediction_summary["probabilities"]
test_Classification_results = prediction_summary["predicted_classes"]

pre_df['GWR_Raw_Score'] = test_raw_scores
pre_df['GWR_Prob'] = test_probabilities
pre_df['GWR_Binary_Class'] = test_Classification_results

print(f"The shape of the true value pre_y of the prediction set:{pre_y.shape}")
print(f"The shape of the original output of the prediction set:{test_raw_scores.shape}")
print(f"The shape of the prediction set probability:{test_probabilities.shape}")
print(f"The shape of the prediction set classification result:{test_Classification_results.shape}")
print(f":{float(test_raw_scores.min()):.6f} ~ {float(test_raw_scores.max()):.6f}")
print(f":{float(test_probabilities.min()):.6f} ~ {float(test_probabilities.max()):.6f}")
print(f":{prediction_summary['probability_threshold']:.4f}")

evaluation = prediction_summary['evaluation']
if evaluation is None:
    print("\\n The current pre_df has no real labels and only outputs raw GWR scores, sigmoid probabilities and classification results.")
elif not evaluation['available']:
    print("\\n prediction set has no finite values available, evaluation skipped.")
else:
    reg = evaluation['regression_metrics']
    print(f"\n prediction set regression R2:{reg['r2']:.4f}")
    print(f"RMSE:{reg['rmse']:.4f}")
    print(f"MAE:{reg['mae']:.4f}")

    if evaluation['binary_metrics'] is None:
        print("\\n prediction set label has only one category, skipping Accuracy / AUC evaluation.")
    else:
        binary = evaluation['binary_metrics']
        print(f"\n prediction set Accuracy:{binary['accuracy']:.4f}")
        print(f"Prediction set ROC-AUC:{binary['roc_auc']:.4f}")
        print("\\n prediction set confusion matrix:")
        print(evaluation['confusion_matrix'])
        print("\\n:")
        print(evaluation['classification_report'])


Implementation note.

Implementation note.

In [ ]:
# part1_save_class_boundaries.py
import numpy as np
import pickle
from pathlib import Path
import jenkspy

CLASS_BOUNDARIES_PATH = Path("class_boundaries.pkl")
CLASSIFICATION_ARTIFACTS_PATH = Path("gwr_classification_artifacts.pkl")
N_CLASSES = 5

import mgtwr.gwr_sigmoid_utils as gwr_sigmoid_utils


def build_jenks_breaks(probabilities, requested_n_classes):
    probabilities = np.asarray(probabilities, dtype=float).reshape(-1)
    finite_values = probabilities[np.isfinite(probabilities)]
    if finite_values.size == 0:
        raise ValueError("Jenks .")

    unique_values = np.unique(finite_values)
    actual_n_classes = int(min(requested_n_classes, unique_values.size))
    if actual_n_classes < 1:
        raise ValueError("The actual number of gradable categories is less than 1.")

    if actual_n_classes == 1:
        only_value = float(unique_values[0])
        class_breaks = np.array([only_value, only_value], dtype=float)
    else:
        class_breaks = np.asarray(jenkspy.jenks_breaks(finite_values, n_classes=actual_n_classes), dtype=float)

    metadata = {
        'requested_n_classes': int(requested_n_classes),
        'actual_n_classes': actual_n_classes,
        'unique_probability_count': int(unique_values.size),
    }
    return class_breaks, metadata


def classify_with_breaks(probabilities, class_breaks):
    probabilities = np.asarray(probabilities, dtype=float).reshape(-1)
    class_breaks = np.sort(np.asarray(class_breaks, dtype=float).reshape(-1))
    labels = np.full(probabilities.shape, -1, dtype=int)
    valid = np.isfinite(probabilities)
    labels[valid] = np.digitize(probabilities[valid], class_breaks[1:-1], right=True)
    labels = np.clip(labels, 0, max(len(class_breaks) - 2, 0))
    return labels


y_pred = np.asarray(y_pred_gwr, dtype=float).reshape(-1)
transform_metadata = None
if CLASSIFICATION_ARTIFACTS_PATH.exists():
    with CLASSIFICATION_ARTIFACTS_PATH.open('rb') as f:
        artifacts = pickle.load(f)
    transform_metadata = artifacts.get('transform_metadata')

y_prob, transform_metadata = gwr_sigmoid_utils.gwr_scores_to_probabilities(y_pred, transform_metadata)
class_breaks, jenks_metadata = build_jenks_breaks(y_prob, N_CLASSES)
susceptibility_labels = classify_with_breaks(y_prob, class_breaks)

print("Probability statistics:")
print(f"Minimum value:{y_prob.min():.4f}")
print(f"Maximum value:{y_prob.max():.4f}")
print(f":{y_prob.mean():.4f}")
print(f"Conversion method:{transform_metadata['method']}")
print(f"Center value (center):{transform_metadata['center']:.4f}")
print(f"Scale:{transform_metadata['scale']:.4f}")
print(f":{jenks_metadata['requested_n_classes']}")
print(f"Actual number of categories:{jenks_metadata['actual_n_classes']}")
print(f"Number of unique probability values:{jenks_metadata['unique_probability_count']}")
print("Classification breakpoint:")
for i, break_point in enumerate(class_breaks):
    print(f"Category Boundary{i}: {break_point:.4f}")

boundary_data = {
    'class_breaks': class_breaks,
    'n_classes': int(jenks_metadata['actual_n_classes']),
    'requested_n_classes': int(jenks_metadata['requested_n_classes']),
    'unique_probability_count': int(jenks_metadata['unique_probability_count']),
    'transform_metadata': transform_metadata,
    'raw_score_stats': {
        'min': float(y_pred.min()),
        'max': float(y_pred.max()),
        'mean': float(y_pred.mean()),
    },
    'probability_stats': {
        'min': float(y_prob.min()),
        'max': float(y_prob.max()),
        'mean': float(y_prob.mean()),
    }
}

with CLASS_BOUNDARIES_PATH.open('wb') as f:
    pickle.dump(boundary_data, f, protocol=pickle.HIGHEST_PROTOCOL)

print("\\n class_boundaries.pkl")

unique, counts = np.unique(susceptibility_labels, return_counts=True)
print("\\n:")
level_names = ['Very low', 'Low', 'Medium', 'High', 'Extremely high']
for label, count in zip(unique, counts):
    level_name = level_names[label] if label < len(level_names) else f'Level{label}'
    print(f"{level_name}: {count}samples ({count/len(y_prob)*100:.1f}%)")


Implementation note.

In [ ]:
# part2_load_and_classify.py
import numpy as np
import pickle
from pathlib import Path

CLASS_BOUNDARIES_PATH = Path("class_boundaries.pkl")

import mgtwr.gwr_sigmoid_utils as gwr_sigmoid_utils


def classify_with_saved_boundaries(probabilities, class_breaks):
    probabilities = np.asarray(probabilities, dtype=float).reshape(-1)
    class_breaks = np.sort(np.asarray(class_breaks, dtype=float).reshape(-1))
    labels = np.full(probabilities.shape, -1, dtype=int)
    valid = np.isfinite(probabilities)
    labels[valid] = np.digitize(probabilities[valid], class_breaks[1:-1], right=True)
    labels = np.clip(labels, 0, len(class_breaks) - 2)
    return labels

with CLASS_BOUNDARIES_PATH.open('rb') as f:
    boundary_data = pickle.load(f)

class_breaks = boundary_data['class_breaks']
transform_metadata = boundary_data.get('transform_metadata')
stats = boundary_data['probability_stats']
actual_n_classes = int(boundary_data.get('n_classes', len(class_breaks) - 1))
requested_n_classes = int(boundary_data.get('requested_n_classes', actual_n_classes))

print("Successfully read category boundary:")
for i, break_point in enumerate(class_breaks):
    print(f"Category Boundary{i}: {break_point:.4f}")

print(f"Probability statistics when \n is saved:")
print(f"Minimum value:{stats['min']:.4f}")
print(f"Maximum value:{stats['max']:.4f}")
print(f":{stats['mean']:.4f}")
print(f":{requested_n_classes}")
print(f"Actual number of categories:{actual_n_classes}")
if transform_metadata is not None:
    print(f"Conversion method:{transform_metadata['method']}")
    print(f"center: {transform_metadata['center']:.4f}")
    print(f"scale: {transform_metadata['scale']:.4f}")

y_pred = np.asarray(y_pred_gwr, dtype=float).reshape(-1)
y_prob, transform_metadata = gwr_sigmoid_utils.gwr_scores_to_probabilities(y_pred, transform_metadata)
susceptibility_labels = classify_with_saved_boundaries(y_prob, class_breaks)

print("Probability statistics of \\n new data:")
print(f"Minimum value:{y_prob.min():.4f}")
print(f"Maximum value:{y_prob.max():.4f}")
print(f":{y_prob.mean():.4f}")

unique, counts = np.unique(susceptibility_labels, return_counts=True)
print("\\n Sample distribution of each category:")
level_names = ['Very low', 'Low', 'Medium', 'High', 'Extremely high']
english_level_names = ['Extremely low', 'Low', 'Moderate', 'High', 'Extremely high']
class_names = {i: english_level_names[i] for i in range(min(actual_n_classes, len(english_level_names)))}

for label, count in zip(unique, counts):
    chinese_name = level_names[label] if label < len(level_names) else f'Level{label}'
    english_name = class_names.get(label, f'Class_{label}')
    percentage = count / len(y_prob) * 100
    print(f"{chinese_name}({english_name}): {count}samples ({percentage:.1f}%)")

pre_df['Susceptibility_Prob'] = y_prob
pre_df['Susceptibility_Class'] = susceptibility_labels
pre_df['Susceptibility_Level'] = pre_df['Susceptibility_Class'].map(class_names).fillna('Unclassified')

print("\\n! Susceptibility_Prob Susceptibility_Class")
